In [1]:
import requests
from IPython.display import JSON
import json


def es_get(url: str, body: dict = None):
    result = requests.get(f"http://elasticsearch:9200/{url}", json=body).json()
    return JSON(result, expanded=True)


def es_post(url: str, body: dict = None):
    result = requests.post(f"http://elasticsearch:9200/{url}", json=body).json()
    return JSON(result, expanded=True)


def es_put(url: str, body: dict):
    result = requests.put(f"http://elasticsearch:9200/{url}", json=body).json()
    return JSON(result, expanded=True)


def es_bulk(lines: list):
    result = requests.put(f"http://elasticsearch:9200/_bulk", data=lines,
                          headers={"Content-Type": "application/json"}).json()
    return JSON(result, expanded=True)

In [2]:
es_post("books*/_delete_by_query",
        {
            "query": {
                "match_all": {}
            }
        })

<IPython.core.display.JSON object>

# Intro

[Elasticsearch](https://www.elastic.co/elasticsearch) is based on [Apache Lucene](https://lucene.apache.org/) which provides the basic building blocks for search, like an inverted index and TF-IDF scoring. Elasticsearch adds on top of that a REST API, index management facilities and additional features like aggregations, query types and suggesters. Additionally, the backing company Elastic provides a whole ecosystem for ingesting and analyzing many different types of documents like log messages.

Besides full-text search capabilities, Elasticsearch acts as a NoSQL database. Documents are organized in indices, and are represented by JSON structures. Document fields can be mapped to different types like text, number, boolean or date. Text fields can be analyzed according to the full-text search requirements.

# Indexing

Add one document.

In [3]:
es_post("books/_doc",
        {
            "name": "Snow Crash",
            "author": "Neal Stephenson",
            "release_date": "1992-06-01",
            "page_count": 470
        })

<IPython.core.display.JSON object>

Add multiple documents.

In [4]:
es_bulk("""
{ "index" : { "_index" : "books" } }
{"name": "Revelation Space", "author": "Alastair Reynolds", "release_date": "2000-03-15", "page_count": 585}
{ "index" : { "_index" : "books" } }
{"name": "1984", "author": "George Orwell", "release_date": "1985-06-01", "page_count": 328}
{ "index" : { "_index" : "books" } }
{"name": "Fahrenheit 451", "author": "Ray Bradbury", "release_date": "1953-10-15", "page_count": 227}
{ "index" : { "_index" : "books" } }
{"name": "Brave New World", "author": "Aldous Huxley", "release_date": "1932-06-01", "page_count": 268}
{ "index" : { "_index" : "books" } }
{"name": "The Handmaids Tale", "author": "Margaret Atwood", "release_date": "1985-06-01", "page_count": 311}
""")

<IPython.core.display.JSON object>

Added documents are not immediately visible after indexing. To make them visible, you can force a refresh.

In [5]:
es_post("books/_refresh")

<IPython.core.display.JSON object>

# Exploring the index with Kibana

1. Open the [Kibana Discovery view](http://127.0.0.1:5601/app/discover).
2. Create a new data view for the index pattern `books` without a timestamp field.

You can now try different things:

* View details of a documents.
* Sort documents.
* Select which fields to show in the list of documents.
* Filters documents using KQL.
* Visualize field statistics.

# Searching

Just return all documents in the index.

In [6]:
es_get("books/_search")

<IPython.core.display.JSON object>

Let's run a simple query to `match` the term `brave` in the field `name`

In [7]:
es_get("books/_search",
       {
           "query": {
               "match": {
                   "name": "brave"
               }
           }
       })

<IPython.core.display.JSON object>

You can also search across multiple fields. By default, at least one search term has to match (`OR`).

In [8]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave george",
                   "fields": ["name", "author"]
               }
           }
       })

<IPython.core.display.JSON object>

You can also require that all search terms must match (`AND`). Use `cross_fields` type, otherwise all search terms must be in the same field.

In [9]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave george",
                   "fields": ["name", "author"],
                   "operator": "and",
                   "type": "cross_fields"
               }
           }
       })

<IPython.core.display.JSON object>

In [10]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave aldous",
                   "fields": ["name", "author"],
                   "operator": "and",
                   "type": "cross_fields"
               }
           }
       })

<IPython.core.display.JSON object>

Fields can also be weighted.

In [11]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave aldous",
                   "fields": ["name^5", "author^1"],
                   "operator": "and",
                   "type": "cross_fields"
               }
           }
       })

<IPython.core.display.JSON object>

Alternative, you can use a compound query.

In [12]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "must": [
                       {
                           "multi_match": {
                               "query": "brave",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       },
                       {
                           "multi_match": {
                               "query": "aldous",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       }
                   ]
               }
           }
       })

<IPython.core.display.JSON object>

The compound query also supports `should` for optional matches that are not required but boost the score when matched.

In [13]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "should": [
                       {
                           "multi_match": {
                               "query": "brave",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       },
                       {
                           "multi_match": {
                               "query": "george",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       }
                   ]
               }
           }
       })

<IPython.core.display.JSON object>

In [14]:
es_get("books/_search",
       {
           "suggest": {
               "text": "brve new word",
               "my_phrase_suggestion": {
                   "phrase": {
                       "field": "name",
                       "size": 1,
                       "gram_size": 3,
                       "direct_generator": [{
                           "field": "name",
                           "suggest_mode": "always",
                           "min_word_length": 1
                       }]
                   }
               }
           }
       })

<IPython.core.display.JSON object>

# Index schema

What does the field mapping look like?

In [15]:
es_get("books/_mapping")

<IPython.core.display.JSON object>

Elasticsearch automatically infers suitable field types based on the field contents during indexing e.g. `name` is `text`, `page_count` is `long` and `release_date` is `date`.

`text` is the default field for (English) full-text search.

Automatically mapped `text` fields have a sub-field named `.keyword` that matches the entire field value.

In [16]:
es_get("books/_search",
       {
           "query": {
               "match": {
                   "name.keyword": "brave"
               }
           }
       })

<IPython.core.display.JSON object>

In [17]:
es_get("books/_search",
       {
           "query": {
               "match": {
                   "name.keyword": "Brave New World"
               }
           }
       })

<IPython.core.display.JSON object>

# Analyzer

In [18]:
es_post("_analyze",
        {
            "analyzer": "standard",
            "text": "The brave new Worlds!"
        })

<IPython.core.display.JSON object>

Use `english` for stemming and stop word removal.

In [19]:
es_post("_analyze",
        {
            "analyzer": "english",
            "text": "The brave new Worlds!"
        })

<IPython.core.display.JSON object>

Or `german` for German-specific analysis.

In [20]:
es_post("_analyze",
        {
            "analyzer": "german",
            "text": "Die tapferen neuen Welten!"}
        )

<IPython.core.display.JSON object>

Use explicit mapping to define fields.

In [21]:
es_put("books_de",
       {
           "mappings": {
               "properties": {
                   "name": {"type": "text", "analyzer": "german"},
                   "author": {"type": "text"},
                   "release_date": {"type": "date"},
                   "page_count": {"type": "long"}
               }
           }
       })

<IPython.core.display.JSON object>

In [22]:
es_post(
    "books_de/_doc",
    {
        "name": "Die tapferen neuen Welten!",
        "author": "Aldous Huxley",
        "release_date": "1932-06-01",
        "page_count": 268,
    },
)

<IPython.core.display.JSON object>

In [23]:
es_get("books_de/_search",
       {
           "query": {
               "match": {
                   "name": "welt"
               }
           }
       })

<IPython.core.display.JSON object>

# Return fields

In [24]:
es_get("books/_search",
       {
           "fields": ["name", "release_date"],
           "_source": False
       })

<IPython.core.display.JSON object>

# Pagination

In [25]:
es_get("books/_search",
       {
           "from": 1,
           "size": 2
       })

<IPython.core.display.JSON object>

# Sorting

In [26]:
es_get("books/_search",
       {
           "sort": [
               {
                   "release_date": {
                       "order": "asc"
                   }
               }
           ]
       })

<IPython.core.display.JSON object>

In [27]:
es_get("books/_search",
       {
           "sort": [
               {
                   "name.keyword": {
                       "order": "asc"
                   }
               }
           ]
       })

<IPython.core.display.JSON object>

# Aggregations

Let's add a `genre` field to the documents, and an `_id` so documents can be updated if needed.

In [28]:
es_post("books/_delete_by_query",
        {
            "query": {
                "match_all": {}
            }
        })

es_bulk("""
{"index": {"_index": "books", "_id": 1}}
{"name": "Revelation Space", "author": "Alastair Reynolds", "release_date": "2000-03-15", "page_count": 585, "genre": "Science Fiction"}
{"index": {"_index": "books", "_id": 2}}
{"name": "1984", "author": "George Orwell", "release_date": "1985-06-01", "page_count": 328, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 3}}
{"name": "Fahrenheit 451", "author": "Ray Bradbury", "release_date": "1953-10-15", "page_count": 227, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 4}}
{"name": "Brave New World", "author": "Aldous Huxley", "release_date": "1932-06-01", "page_count": 268, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 5}}
{"name": "The Handmaids Tale", "author": "Margaret Atwood", "release_date": "1985-06-01", "page_count": 311, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 6}}
{"name": "The Hobbit", "author": "J.R.R. Tolkien", "release_date": "1937-09-21", "page_count": 310, "genre": "Fantasy"}
{"index": {"_index": "books", "_id": 7}}
{"name": "A Game of Thrones", "author": "George R.R. Martin", "release_date": "1996-08-06", "page_count": 694, "genre": "Fantasy"}
{"index": {"_index": "books", "_id": 8}}
{"name": "Hyperion", "author": "Dan Simmons", "release_date": "1989-05-26", "page_count": 482, "genre": "Science Fiction"}
{"index": {"_index": "books", "_id": 9}}
{"name": "Dune", "author": "Frank Herbert", "release_date": "1965-08-01", "page_count": 412, "genre": "Science Fiction"}
""")

es_post("books/_refresh")

<IPython.core.display.JSON object>

In [29]:
es_get("books/_search",
       {
           "size": 0,
           "aggs": {
               "genres": {
                   "terms": {
                       "field": "genre.keyword"
                   }
               }
           }
       })

<IPython.core.display.JSON object>

In [30]:
es_get("books/_search",
       {
           "size": 0,
           "aggs": {
               "page_ranges": {
                   "range": {
                       "field": "page_count",
                       "ranges": [
                           {"from": 0, "to": 100},
                           {"from": 101, "to": 200},
                           {"from": 201, "to": 300},
                           {"from": 301, "to": 400},
                           {"from": 401, "to": 500}
                       ]
                   }
               }
           }
       })

<IPython.core.display.JSON object>

# Filters

In [31]:
es_get("books/_search",
       {
           "query": {
               "term": {
                   "genre.keyword": {
                       "value": "Dystopian"
                   }
               }
           }
       })

<IPython.core.display.JSON object>

In [32]:
es_get("books/_search",
       {
           "query": {
               "range": {
                   "page_count": {
                       "gte": 500
                   }
               }
           }
       })

<IPython.core.display.JSON object>

Filters can also be expressed as compound queries.

In [33]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "filter": [
                       {
                           "term": {
                               "genre.keyword": {
                                   "value": "Dystopian"
                               }
                           }
                       }
                   ]
               }
           }
       })

<IPython.core.display.JSON object>

And combined with optional search terms for boosting matching results.

In [34]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "filter": [
                       {
                           "term": {
                               "genre.keyword": {
                                   "value": "Dystopian"
                               }
                           }
                       }
                   ],
                   "should": [
                       {
                           "multi_match": {
                               "query": "brave",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       }
                   ],
               }
           }
       })

<IPython.core.display.JSON object>

# Debugging

In [35]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave george",
                   "fields": ["name", "author"],
               }
           },
           "explain": True
       })

<IPython.core.display.JSON object>

In [36]:
es_get("books/_explain/4",
       {
           "query": {
               "match": {
                   "name": "world"
               }
           }
       })

<IPython.core.display.JSON object>

In [37]:
es_get("books/_explain/1",
       {
           "query": {
               "match": {
                   "name": "world"
               }
           }
       })

<IPython.core.display.JSON object>